# 03 — Data Preparation and Master Dataset
Executes the complete integration and validation pipeline.

In [4]:
# import libraries
from pathlib import Path
import sys
import yaml


In [6]:
# set project root and config path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'data_integration.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk/configs/data_integration.yaml')

In [7]:
PROJECT_ROOT

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/Codigo/ontario-electricity-peak-risk')

In [8]:
# load config
with CONFIG_PATH.open('r', encoding='utf-8') as file:
    config = yaml.safe_load(file)
config

{'analysis_period': {'start_year': 2021, 'end_year': 2025},
 'paths': {'processed_data_dir': 'data/processed',
  'master_dataset': 'data/processed/master_dataset.parquet',
  'reports_dir': 'reports/data_preparation',
  'docs_dir': 'docs'},
 'fsa_station_mapping': {'M5S': {'climate_id': 6158355,
   'station_name': 'TORONTO CITY'},
  'M5R': {'climate_id': 6158355, 'station_name': 'TORONTO CITY'},
  'M6G': {'climate_id': 6158355, 'station_name': 'TORONTO CITY'},
  'L4T': {'climate_id': 6158731, 'station_name': 'TORONTO INTL A'},
  'M9W': {'climate_id': 6158731, 'station_name': 'TORONTO INTL A'},
  'M9R': {'climate_id': 6158731, 'station_name': 'TORONTO INTL A'}}}

In [11]:
# verify config
from pathlib import Path

processed_dir = PROJECT_ROOT / "data" / "processed"

print(processed_dir)
print("Exists:", processed_dir.exists())
print("Files found:")

for file in processed_dir.glob("*"):
    print(file.name)

e:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\Codigo\ontario-electricity-peak-risk\data\processed
Exists: True
Files found:
.gitkeep
calendar_hourly_ontario_timestamp.csv
hourly_consumption_fsa_l4t.csv
hourly_consumption_fsa_m5r.csv
hourly_consumption_fsa_m5s.csv
hourly_consumption_fsa_m6g.csv
hourly_consumption_fsa_m9r.csv
hourly_consumption_fsa_m9w.csv
master_dataset.parquet
weather_toronto_city_6158355.csv
weather_toronto_intl_a_6158731.csv


In [12]:
# import build_master_dataset function
from src.ontario_peak_risk.data_preparation.build_master_dataset import build_master_dataset

In [14]:
# build master dataset
master_dataset = build_master_dataset(CONFIG_PATH)
master_dataset.shape

Created: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\Codigo\ontario-electricity-peak-risk\data\processed\master_dataset.parquet | rows=262,944 | columns=71


(262944, 71)

In [16]:
{
 'rows': len(master_dataset),
 'columns': master_dataset.shape[1],
 'fsas': sorted(master_dataset['fsa'].unique().tolist()),
 'minimum_timestamp': master_dataset['timestamp'].min(),
 'maximum_timestamp': master_dataset['timestamp'].max(),
 'duplicate_key_rows': int(master_dataset.duplicated(['fsa','timestamp'], keep=False).sum()),
}

{'rows': 262944,
 'columns': 71,
 'fsas': ['L4T', 'M5R', 'M5S', 'M6G', 'M9R', 'M9W'],
 'minimum_timestamp': Timestamp('2021-01-01 00:00:00'),
 'maximum_timestamp': Timestamp('2025-12-31 23:00:00'),
 'duplicate_key_rows': 0}

In [17]:
master_dataset.groupby('fsa').agg(rows=('timestamp','size'), unique_hours=('timestamp','nunique'), min_timestamp=('timestamp','min'), max_timestamp=('timestamp','max')).reset_index()

,fsa,rows,unique_hours,min_timestamp,max_timestamp
0,L4T,43824,43824,2021-01-01,2025-12-31 23:00:00
1,M5R,43824,43824,2021-01-01,2025-12-31 23:00:00
2,M5S,43824,43824,2021-01-01,2025-12-31 23:00:00
3,M6G,43824,43824,2021-01-01,2025-12-31 23:00:00
4,M9R,43824,43824,2021-01-01,2025-12-31 23:00:00
5,M9W,43824,43824,2021-01-01,2025-12-31 23:00:00


## Summary
Review: 
- Report under `docs/Data_Preparation_Report.md`.
- Removed columns under `docs/Removed_Weather_Columns.md`.
- CSV reports under `reports/data_preparation/`.